In [18]:
import pandas as pd
import json
import ast
from pymongo import MongoClient

In [17]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["censo_locales_db"]
collection = db["locales"]

# Función genérica para cargar JSON

In [3]:
def cargar_json(ruta) -> pd.DataFrame:
    with open(ruta, "r", encoding="utf-8") as f:
        data = json.load(f)

    return pd.DataFrame(data)

In [16]:
df_airbnb = cargar_json("../doc_files/airbnb/airbnb_listings.json")

def explorar_dataset(nombre, df, output_md=True, ruta_salida=None):
    """
    Genera un reporte de exploración en Markdown ordenado y legible
    """

    md = []

    # Título
    md.append(f"# 📊 Exploración Dataset: {nombre}\n")

    # Información general
    md.append("## Información general\n")
    md.append(f"- **Registros:** {df.shape[0]:,}")
    md.append(f"- **Columnas:** {df.shape[1]}")
    md.append("")

    # Columnas
    md.append("## Columnas\n")
    columnas = pd.DataFrame({
        "columna": df.columns,
        "tipo": df.dtypes.astype(str)
    })
    md.append(columnas.to_markdown(index=False))
    md.append("")

    # Nulos
    md.append("## Valores nulos\n")
    nulos = df.isnull().sum().reset_index()
    nulos.columns = ["columna", "nulos"]
    nulos = nulos.sort_values("nulos", ascending=False)
    md.append(nulos.to_markdown(index=False))
    md.append("")

    # Duplicados
    md.append("## Registros duplicados\n")

    df_tmp = df.copy()

    for col in df_tmp.columns:
        df_tmp[col] = df_tmp[col].apply(
            lambda x: str(x) if isinstance(x, (list, dict)) else x
        )

    duplicados = df_tmp.duplicated().sum()
    md.append(f"- **Duplicados:** {duplicados}")
    md.append("")

    # Head
    md.append("## Primeras filas (head)\n")
    md.append(df.head().to_markdown(index=False))
    md.append("")

    # Describe numérico
    md.append("## Estadísticas numéricas\n")
    md.append(df.describe().to_markdown())
    md.append("")

    # Describe categórico
    md.append("## Estadísticas categóricas\n")
    md.append(df.describe(include="object").to_markdown())
    md.append("")

    contenido = "\n".join(md)

    # Guardar archivo
    if output_md and ruta_salida:
        with open(ruta_salida, "w", encoding="utf-8") as f:
            f.write(contenido)

    return contenido

# ==========================
# LISTA DE DATASETS
# ==========================

datasets = {
    "airbnb": df_airbnb,
}

for nombre, df in datasets.items():
    explorar_dataset(
        nombre.capitalize(),
        df,
        ruta_salida=f"reporte_{nombre}.md"
    )

C:\Users\braya\AppData\Local\Temp\ipykernel_40388\4103157634.py:62: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  md.append(df.describe(include="object").to_markdown())


# CARGA DE DATOS AIRBNB

In [21]:
collection = db["alojamientos_airbnb"]

In [19]:
print("Registros originales:", len(df_airbnb))

Registros originales: 16313


In [20]:
df_airbnb["beds"] = df_airbnb["beds"].fillna(0)
df_airbnb["bedrooms"] = df_airbnb["bedrooms"].fillna(0)
df_airbnb["name"] = df_airbnb["name"].fillna("Sin nombre")

In [ ]:
def parse_location(loc):
    """
    Convierte el string de ubicación a GeoJSON válido
    """
    try:
        data = ast.literal_eval(loc)
        coords = data.get("coordinates", [None, None])
        return {
            "type": "Point",
            "coordinates": coords
        }
    except:
        return None


df_airbnb["ubicacion"] = df_airbnb["location"].apply(parse_location)

def parse_amenities(x):
    try:
        return ast.literal_eval(x)
    except:
        return []


df_airbnb["amenities"] = df_airbnb["amenities"].apply(parse_amenities)

df_final = pd.DataFrame({
    "airbnb_id": df_airbnb["id"],
    "host_id": df_airbnb["host_id"],
    "nombre": df_airbnb["name"],
    "distrito": df_airbnb["neighbourhood_group_cleansed"],
    "capacidad": df_airbnb["accommodates"],
    "habitaciones": df_airbnb["bedrooms"],
    "camas": df_airbnb["beds"],
    "precio": df_airbnb["price"],
    "tipo_habitacion": df_airbnb["room_type"],
    "numero_reviews": df_airbnb["number_of_reviews"],
    "amenities": df_airbnb["amenities"],
    "ubicacion": df_airbnb["ubicacion"]
})

# datos de listo para mongo

datos = df_final.to_dict(orient="records")


In [ ]:
collection.delete_many({}) 

collection.insert_many(datos)

print("Datos insertados:", len(datos))

collection.create_index([("distrito", 1)])
collection.create_index([("ubicacion", "2dsphere")])

print("Indices creados")

✅ Datos insertados correctamente: 16313
✅ Índices creados


In [24]:
doc = collection.find_one()

import pprint
pprint.pprint(doc)

{'_id': ObjectId('69965abd8cca00326b44cb48'),
 'airbnb_id': 18628,
 'amenities': [],
 'camas': 1.0,
 'capacidad': 2,
 'distrito': 'Centro',
 'habitaciones': 0.0,
 'host_id': 71597,
 'nombre': 'Greta Studio Wifi Chueca en Madrid',
 'numero_reviews': 37,
 'precio': 54.0,
 'tipo_habitacion': 'Entire home/apt',
 'ubicacion': None}


In [25]:
from collections import defaultdict

def infer_schema(collection, sample_size=100):
    schema = defaultdict(set)

    cursor = collection.find().limit(sample_size)

    for doc in cursor:
        for key, value in doc.items():
            schema[key].add(type(value).__name__)

    print("📊 Esquema inferido:\n")

    for campo, tipos in schema.items():
        print(f"{campo}: {', '.join(tipos)}")


infer_schema(collection)


📊 Esquema inferido:

_id: ObjectId
airbnb_id: int
host_id: int
nombre: str
distrito: str
capacidad: int
habitaciones: float
camas: float
precio: float
tipo_habitacion: str
numero_reviews: int
amenities: list
ubicacion: NoneType


In [26]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["censo_locales_db"]

collection_locales = db["locales"]
collection_airbnb = db["alojamientos_airbnb"]


In [27]:
collection_alojamientos = db["alojamientos"]

for doc in collection_airbnb.find():

    nuevo_doc = {
        "airbnb_id": doc.get("airbnb_id"),
        "nombre": doc.get("nombre"),

        "ubicacion": {
            "distrito": (doc.get("distrito") or "").strip().upper() or None,
            "barrio": None,
            "coordenadas": None
        },

        "propiedad": {
            "tipo_habitacion": doc.get("tipo_habitacion"),
            "capacidad": doc.get("capacidad"),
            "habitaciones": doc.get("habitaciones"),
            "camas": doc.get("camas")
        },

        "precio": doc.get("precio"),
        "numero_reviews": doc.get("numero_reviews"),

        "host": {
            "host_id": doc.get("host_id")
        },

        "amenities": doc.get("amenities", [])
    }

    collection_alojamientos.insert_one(nuevo_doc)


In [28]:
collection_alojamientos.create_index([("ubicacion.distrito", 1)])
collection_alojamientos.create_index([("precio", 1)])
collection_alojamientos.create_index([("amenities", 1)])

collection_locales.create_index([("local.desc_distrito_local", 1)])
collection_locales.create_index([("local.desc_barrio_local", 1)])


'local.desc_barrio_local_1'

In [29]:
for doc in collection_locales.find():

    distrito = doc.get("local", {}).get("desc_distrito_local")
    barrio = doc.get("local", {}).get("desc_barrio_local")

    if distrito:
        distrito = distrito.strip().upper()

    if barrio:
        barrio = barrio.strip().upper()

    collection_locales.update_one(
        {"_id": doc["_id"]},
        {
            "$set": {
                "local.desc_distrito_local": distrito,
                "local.desc_barrio_local": barrio
            }
        }
    )


In [30]:
pipeline = [
    {
        "$group": {
            "_id": "$ubicacion.distrito",
            "num_alojamientos": {"$sum": 1},
            "precio_medio": {"$avg": "$precio"}
        }
    }
]

stats = list(collection_alojamientos.aggregate(pipeline))


In [31]:
stats_dict = {
    s["_id"]: s for s in stats if s["_id"] is not None
}


In [32]:
for doc in collection_locales.find():

    distrito = doc.get("local", {}).get("desc_distrito_local")

    if distrito in stats_dict:

        data = stats_dict[distrito]

        contexto = {
            "distrito": distrito,
            "num_alojamientos_cercanos": data["num_alojamientos"],
            "precio_medio_alojamientos": data["precio_medio"]
        }

        collection_locales.update_one(
            {"_id": doc["_id"]},
            {"$set": {"contexto_turistico": contexto}}
        )


In [34]:
# Locales en zonas turísticas
result = collection_locales.find({
    "contexto_turistico.num_alojamientos_cercanos": {"$gt": 50}
})

for r in result:
    print(r["_id"])


20000596
20000605
20000669
20000709
20000721
20000729
20000756
20000761
20000764
20000766
20000783
20000797
20000811
20000813
20000864
20000880
20000899
20000902
20000903
20000920
20000937
20000939
20000948
20000956
20000958
20000959
20001008
20001013
20001015
20001003
20001010
20001027
20001029
20001031
20001049
20001053
20001057
20001080
20001084
20001085
20001088
20000767
20000793
20000795
20000809
20000814
20000818
20000822
20000824
20000826
20000829
20000844
20000852
20000887
10000004
10000116
10000150
10000224
10000264
10000412
10000413
10000442
10000455
10000462
20000904
20000926
20000944
20000946
20000965
20000974
20000985
20000990
20000998
20001016
10000463
10000478
10000003
10000044
10000097
10000102
10000162
10000385
10000398
10000401
20001028
20001102
20001104
20001107
20001109
20001124
20001127
20001159
20001179
20001186
10000422
10000451
10000503
10000534
10000013
10000052
10000071
10000105
10000226
10000275
20001191
20001195
20001216
20001038
20001058
20001062
20001071
2

In [ ]:
# Precio medio por distrito

pipeline = [
    {
        "$group": {
            "_id": "$ubicacion.distrito",
            "precio_medio": {"$avg": "$precio"}
        }
    }
]

for r in collection_alojamientos.aggregate(pipeline):
    print(r)

{'_id': 'LATINA', 'precio_medio': 43.30641330166271}
{'_id': 'VILLAVERDE', 'precio_medio': 45.73118279569893}
{'_id': 'MONCLOA - ARAVACA', 'precio_medio': 84.29376257545272}
{'_id': 'BARAJAS', 'precio_medio': 43.40869565217391}
{'_id': 'CENTRO', 'precio_medio': 81.26728817559594}
{'_id': 'CHAMBERÍ', 'precio_medio': 85.97953216374269}
{'_id': 'CHAMARTÍN', 'precio_medio': 84.7860465116279}
{'_id': 'CIUDAD LINEAL', 'precio_medio': 43.980246913580245}
{'_id': 'RETIRO', 'precio_medio': 83.43843283582089}
{'_id': 'SAN BLAS - CANILLEJAS', 'precio_medio': 49.00595238095238}
{'_id': 'TETUÁN', 'precio_medio': 58.23116438356164}
{'_id': 'CARABANCHEL', 'precio_medio': 38.7417943107221}
{'_id': 'VILLA DE VALLECAS', 'precio_medio': 44.083333333333336}
{'_id': 'HORTALEZA', 'precio_medio': 61.721739130434784}
{'_id': 'FUENCARRAL - EL PARDO', 'precio_medio': 62.27225130890052}
{'_id': 'VICÁLVARO', 'precio_medio': 41.78787878787879}
{'_id': 'PUENTE DE VALLECAS', 'precio_medio': 36.71335504885994}
{'_id'